In [14]:
import sys
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import json
from utils.schema_json import ReportData


Added to sys.path: C:\Users\lucat\PythonRepositories\PRIN


In [15]:
# Parameters
TRAIN_FILE_NAME = "data_finetune_guido_openai_train.jsonl"
VALIDATION_FILE_NAME = "data_finetune_guido_openai_val.jsonl"
TIPO = 'openai'

In [ ]:
# Carichiamo i nostri file JSON
paths = [Path('../data/ft-dataset/' + file_name) for file_name in (TRAIN_FILE_NAME, VALIDATION_FILE_NAME)]

data = []
for i, path in enumerate(paths):
    with open(path, "r", encoding="utf-8") as f:
        data_list = [json.loads(line) for line in f]
        data.append(data_list)

train_data, validation_data = data[0], data[1]

print(f"{len(train_data) = }")
print(f"{train_data[0] = }")
print(f"{type(train_data[0]) = }")
print(f"{type(train_data[0]['messages'][2]['content']) = }")

if TIPO == 'openai':
    for dataset in data:
        for record in dataset:
            record['messages'][2]['content'] = ReportData.model_validate_json(record['messages'][2]['content'])
            
print(f"{len(train_data) = }")
print(f"{train_data[0] = }")
print(f"{type(train_data[0]) = }")
print(f"{type(train_data[0]['messages'][2]['content']) = }")

#train_data[0]['messages'][2]['content'].model_dump()

len(train_data) = 116
train_data[0] = {'messages': [{'role': 'system', 'content': 'Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).\n\nIl tuo compito è estrarre le caratteristiche strutturate dal referto RM fornito e mappare i dati nello schema JSON sottostante.\n\nRegole di output rigorose:\n\n1. Restituisci esclusivamente un oggetto JSON valido. Nessun testo introduttivo, spiegazione, codice o commento fuori dal JSON.\n2. Rispetta esattamente i tipi e i valori consentiti per ciascun campo (vedi elenco sotto).\n  - Se un campo è numerico ma nel testo non compare un valore, scrivi null.\n  - Se un campo ha opzioni predefinite ma non è chiaramente determinabile, scrivi null.\n3. I campi booleani all’interno di posizione, infiltrazione_organi_dettagli e sedi_linfonodi  devono contenere solo true o false.\n4. Non aggiungere o inventare informazioni non presenti nel referto. Se un dato manca o è ambiguo, scrivi null.\n\n

Now that we have loaded our dataset, we will convert it to the proper desired format to upload for training.

The data will be converted to a jsonl format as follows:

*THIS IS AN EXAMPLE*
```json
{"text": "Avena e nocciole cioccolato fondente", "labels": {"food": ["sweet-snacks"], "country_label": "italy"}}
{"text": "Pomodori in pezzi", "labels": {"food": ["plant-based-foods-and-beverages"], "country_label": "belgium"}}
{"text": "Grandyoats, Nori Sesame Cashews", "labels": {"food": ["snacks"], "country_label": "united-states"}}
{"text": "Jus d'orange Profit", "labels": {"food": ["beverages", "plant-based-foods-and-beverages"], "country_label": "switzerland"}}
{"text": "Rote Beete", "labels": {"food": ["plant-based-foods", "plant-based-foods-and-beverages"], "country_label": "germany"}}
...
```
With an example of a label being:
```json
"labels": {
  # Chekbox like sedi_linfonodi
  "food": [
    "beverages",
    "plant-based-foods-and-beverages"
  ],
  # Drop-down like morfologia
  "country_label": "switzerland"
}
```
For multi-target classification.

{'morfologia': 'solido_anulare',
 'posizione': {'basso': True,
  'medio': False,
  'alto': False,
  'giunzione': False},
 'ore_inizio': 12,
 'ore_fine': 12,
 'spessore_parietale': None,
 'estensione_cranio_caudale': 60,
 'distanza_oai': 0.0,
 'riflessione_peritoneale_anteriore': 'sotto',
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': None,
 'infiltrazione_organi_extra': 'si',
 'infiltrazione_organi_dettagli': {'pavimento_pelvico': True, 'altro': False},
 'coinvolgimento_riflessione_peritoneale': None,
 'coinvolgimento_fascia_mesorettale': None,
 'numero_linfonodi_non_conosciuto': True,
 'linfonodi_sospetti': 0,
 'sedi_linfonodi': {'mesorettali': False,
  'rettali_superiori': True,
  'otturatori': False,
  'iliaci': False,
  'altro': False},
 'depositi_tumorali': None,
 'numero_depositi': 0,
 'emvi_esteso': 'no',
 'stadio_T': 'T4b',
 'stadio_N': 'N+',
 'stadio_N1c': False,
 'mrf': None,
 'emvi': None,
 'metastasi': None}

In [14]:
ReportData.model_validate(train_data[0]['messages'][2]['content'])

ReportData(morfologia='solido_anulare', posizione=Posizione(basso=True, medio=False, alto=False, giunzione=False), ore_inizio=12, ore_fine=12, spessore_parietale=None, estensione_cranio_caudale=60, distanza_oai=0.0, riflessione_peritoneale_anteriore='sotto', infiltrazione_tessuto_adiposo='si_5mm_plus', infiltrazione_sfinteri=None, infiltrazione_organi_extra='si', infiltrazione_organi_dettagli=InfiltrazioneOrganiDettagli(pavimento_pelvico=True, altro=False), coinvolgimento_riflessione_peritoneale=None, coinvolgimento_fascia_mesorettale=None, numero_linfonodi_non_conosciuto=True, linfonodi_sospetti=0, sedi_linfonodi=SediLinfonodi(mesorettali=False, rettali_superiori=True, otturatori=False, iliaci=False, altro=False), depositi_tumorali=None, numero_depositi=0, emvi_esteso='no', stadio_T='T4b', stadio_N='N+', stadio_N1c=False, mrf=None, emvi=None, metastasi=None)